<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm1-t02-colab-resume/notebooks/m1_pipeline_thu_thap_va_kiem_toan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Mở trong Colab"/></a>

# Thu thập M1 theo hai pha độc lập

Notebook dùng cùng một danh sách tất định 5.000 ID theo MSSV, chia 50 cohort × 100 ID. Chạy lần lượt **cấu hình → arXiv → Semantic Scholar → kiểm toán/xuất**. Mỗi pha có checkpoint riêng nên có thể chạy lại S2 mà không tải lại source, hoặc S2 trước arXiv. Mỗi kết quả mang fingerprint để kiểm tra phạm vi có khớp hay không.

Chọn CPU và kết nối Drive. Mặc định `COHORT_LIMIT = None` xử lý đủ 5.000 vị trí được giao; đặt `1` cho pilot 100 ID. Chế độ pilot tự dùng root Drive/local riêng, còn lượt đầy đủ dùng root corpus hiện có; giữ nguyên root khi resume. Không mở hai runtime ghi cùng MSSV. arXiv là nguồn xác nhận bài và phiên bản; kết quả S2 `not_found` không kết luận arXiv không tồn tại.


## 1. Cấu hình, Drive và mã runtime


In [ ]:
# Đổi pin trong commit notebook sau khi commit runtime đã kiểm thử.
import os
from pathlib import Path
import re
import subprocess
import sys

M1_RUNTIME_COMMIT = "41c62a3e6b721640b2e395a4590ed2eaea1addbe"
MSSV = ""  # Nhập MSSV trong config/group10_id_ranges.json.
COHORT_LIMIT = None  # None: 50 cohort; 1..50: số cohort đầu tiên.
PILOT_ROOT = COHORT_LIMIT is not None
DRIVE_BASE = Path("/content/drive/MyDrive/M1-pilot" if PILOT_ROOT else "/content/drive/MyDrive/M1")
LOCAL_BASE = Path("/content/m1-pilot-local" if PILOT_ROOT else "/content/m1-local")

loaded_src = sorted(name for name in sys.modules if name == "src" or name.startswith("src."))
if loaded_src:
    raise RuntimeError("Kernel đã nạp src trước khi checkout. Chọn Runtime > Restart session rồi chạy lại cell này.")
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn runtime chỉ dùng CPU trong Runtime settings.")
if not MSSV.strip():
    raise ValueError("Nhập MSSV trước khi tiếp tục.")

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/Arxiv-Project-Group10")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if not re.fullmatch(r"[0-9a-f]{40}", M1_RUNTIME_COMMIT):
    raise RuntimeError("M1_RUNTIME_COMMIT phải có đủ 40 ký tự SHA.")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", M1_RUNTIME_COMMIT], check=True)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if current_head != M1_RUNTIME_COMMIT:
    raise RuntimeError("Mã runtime không khớp SHA đã ghim.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.m1_workflow import build_member_selection
SELECTION = build_member_selection(MSSV, COHORT_LIMIT)
DRIVE_ROOT = DRIVE_BASE / MSSV
LOCAL_ROOT = LOCAL_BASE / MSSV
print(f"Runtime: {current_head}")
print(f"MSSV: {MSSV}; IDs: {SELECTION['expected_count']}; cohorts: {len(SELECTION['cohorts'])}")
print(f"Selection fingerprint: {SELECTION['fingerprint']}")
print(f"Drive root: {DRIVE_ROOT}; local root: {LOCAL_ROOT}")


## 2. Pha arXiv: xác minh ID và tải mọi phiên bản

Pha này chỉ ghi bằng chứng arXiv và archive; nó không chứng nhận M1 hoàn tất. Stop hoặc lỗi 429 được lưu để chạy tiếp cell này sau.


In [ ]:
from src.m1_workflow import run_m1_arxiv_member

ARXIV_RESULT = run_m1_arxiv_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    cohort_limit=COHORT_LIMIT,
)
print({key: ARXIV_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "source_complete_papers", "stopped", "stop_reason", "m1_complete",
)})


## 3. Pha Semantic Scholar: metadata và toàn bộ references

S2 chạy độc lập trên cùng danh sách ID, kể cả khi arXiv chưa chạy. Checkpoint giữ metadata, references thô, phân trang, lỗi và checksum. Nếu dừng hoặc gặp 429, chạy lại cell này sau khi dịch vụ ổn định.


In [ ]:
from google.colab import userdata
from src.m1_workflow import run_m1_s2_member

try:
    S2_API_KEY = userdata.get("SEMANTIC_SCHOLAR_API_KEY")
except Exception as exc:
    print(f"Không đọc được Colab Secret ({type(exc).__name__}); lượt S2 dùng quota không xác thực.")
    S2_API_KEY = None

S2_RESULT = run_m1_s2_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    semantic_scholar_api_key=S2_API_KEY,
    cohort_limit=COHORT_LIMIT,
)
S2_API_KEY = None
print({key: S2_RESULT.get(key) for key in (
    "stage", "selection_fingerprint", "expected_id_count",
    "metadata_complete", "references_complete", "stopped", "stop_reason", "m1_complete",
)})


## 4. Kiểm toán, đối soát và xuất bảng

Chỉ chạy cell này sau khi hai pha có đủ bằng chứng. Kiểm toán đối chiếu fingerprint, định danh, checksum và phân trang; chỉ nhập S2 vào corpus v2 khi không có xung đột. Bảng chỉ được xuất nếu toàn bộ phạm vi đạt M1.


In [ ]:
from src.m1_workflow import run_m1_audit_member

AUDIT_RESULT = run_m1_audit_member(
    MSSV,
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    cohort_limit=COHORT_LIMIT,
)
print({key: AUDIT_RESULT.get(key) for key in (
    "selection_fingerprint", "expected_id_count", "merged_s2_ids",
    "is_reconciled", "integrity_failed_papers", "unprocessed_papers",
    "export_status", "export_error", "report_path", "benchmark_summary_path",
)})


## Tiếp tục và benchmark

Khi S2 chạy trước arXiv, quay lại cell arXiv rồi chạy lại cell kiểm toán. Mỗi lần chạy tiếp tục từ checkpoint cùng fingerprint; checkpoint không tương thích hoặc checksum sai được giữ lại để kiểm tra, không tự xóa. Giữ log từng phiên; thời gian retry và delay được tính vào runtime, khoảng nghỉ giữa phiên được loại khỏi tổng thời gian hoạt động.

Pilot 100 ID chỉ xác nhận cách chạy và resume. Benchmark cuối cần chạy CPU Colab trên đủ 20.000 ID được giao cho nhóm; book chỉ dùng số liệu thật đã kiểm toán. Xem [hướng dẫn Colab](../docs/guides/colab.md) và [hướng dẫn tổng hợp benchmark](../docs/operations/benchmark_runner.md).
